# Importers Local Directory Test

导入 Windows 图片目录 `C:\\Users\\wuchaoli\\Pictures\\测试图片`，并创建一个新的本地受管图片库。Notebook 运行产物统一写入 `notebooks/.importers_test_library/`。

In [1]:
from pathlib import Path

from image_gallery.importers import ImportPipeline, LocalDirectoryReader
from image_gallery.storage import FileSystemStorage


repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

windows_source_dir = Path(r"C:\Users\wuchaoli\Pictures\测试图片")
wsl_source_dir = Path("/mnt/c/Users/wuchaoli/Pictures/测试图片")
source_dir = wsl_source_dir if wsl_source_dir.exists() else windows_source_dir
if not source_dir.exists():
    raise FileNotFoundError(f"source directory not found: {source_dir}")

library_root = repo_root / "notebooks" / ".importers_test_library"
storage = FileSystemStorage(storage_name="test_picture_library").connect(root=library_root / "storage")
records = LocalDirectoryReader(source_dir).read()

print("source_dir:", source_dir)
print("library_root:", library_root)
print("storage_name:", storage.storage_name)
print("source records:", len(records))

source_dir: /mnt/c/Users/wuchaoli/Pictures/测试图片
library_root: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library
storage_name: test_picture_library
source records: 33


In [2]:
result = ImportPipeline(
    storage=storage,
    output_dir=library_root / "outputs",
    global_tags=["dataset/test_pictures", "source/windows_pictures"],
).run(records)

print("raw_dataset_path:", result.raw_dataset_path)
print("import_report_path:", result.import_report_path)
print("failure_manifest_path:", result.failure_manifest_path)
print("report:", result.report)

raw_dataset_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/outputs/raw.parquet
import_report_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/outputs/import_report.json
failure_manifest_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/outputs/failure_manifest.jsonl
report: {'success_count': 33, 'failure_count': 0}


In [3]:
import pandas as pd

from image_gallery.dataset import Dataset


raw_dataset = Dataset.from_path(result.raw_dataset_path)
raw_frame = raw_dataset.to_frame()
failure_frame = pd.read_json(result.failure_manifest_path, lines=True) if Path(result.failure_manifest_path).stat().st_size else pd.DataFrame()

print("raw count:", raw_dataset.count())
print("fingerprint:", raw_dataset.fingerprint())
print("failure count:", len(failure_frame))
raw_frame.head()

raw count: 33
fingerprint: ca6d7c13033214e905a8cc90d4c3d434e11768a3903a6ab19a8463a19d3bceac
failure count: 0


,image_id,source_uri,source_type,source_file_name,storage_name,image_uri,import_status,imported_at,schema_version,tags,...,icc_profile_present,dpi_x,dpi_y,exif_orientation,exif_datetime,camera_make,camera_model,gps_present,gps_latitude,gps_longitude
0,69f95b9f-6143-4dbd-8e1f-0b5c32b2ced1,/mnt/c/Users/wuchaoli/Pictures/测试图片/1364137053...,local_directory,13641370531_1943292800.jpg,test_picture_library,/home/wuchaoli/codespace/ImageGallery/notebook...,imported,2026-07-02T09:52:18.902924+00:00,raw.v1,"[dataset/test_pictures, source/windows_pictures]",...,False,NaN,NaN,NaN,None,None,None,False,NaN,NaN
1,83f29c1e-47f3-4e4b-aebf-8e4047cc39c4,/mnt/c/Users/wuchaoli/Pictures/测试图片/1663846702...,local_directory,1663846702686_632c492eb40eef4c936fb55b.jpeg,test_picture_library,/home/wuchaoli/codespace/ImageGallery/notebook...,imported,2026-07-02T09:52:18.940865+00:00,raw.v1,"[dataset/test_pictures, source/windows_pictures]",...,False,NaN,NaN,NaN,None,None,None,False,NaN,NaN
2,b4469a7b-b77a-42c7-9a41-ee70d032bee2,/mnt/c/Users/wuchaoli/Pictures/测试图片/3589c17afe...,local_directory,3589c17afeb34b69ef175659826aa567.jpg,test_picture_library,/home/wuchaoli/codespace/ImageGallery/notebook...,imported,2026-07-02T09:52:18.965227+00:00,raw.v1,"[dataset/test_pictures, source/windows_pictures]",...,False,NaN,NaN,NaN,None,None,None,False,NaN,NaN
3,f909bd64-851d-43fc-a8ec-f4797c47190d,/mnt/c/Users/wuchaoli/Pictures/测试图片/3a2c17fc30...,local_directory,3a2c17fc30c7e68399074e75aa3c79bc.jpg,test_picture_library,/home/wuchaoli/codespace/ImageGallery/notebook...,imported,2026-07-02T09:52:18.992348+00:00,raw.v1,"[dataset/test_pictures, source/windows_pictures]",...,False,NaN,NaN,NaN,None,None,None,False,NaN,NaN
4,0ed988cf-1dfd-44f2-af54-b11012b4d564,/mnt/c/Users/wuchaoli/Pictures/测试图片/49c113e21d...,local_directory,49c113e21d837d234ea57bd48f8751af.png,test_picture_library,/home/wuchaoli/codespace/ImageGallery/notebook...,imported,2026-07-02T09:52:19.020063+00:00,raw.v1,"[dataset/test_pictures, source/windows_pictures]",...,False,NaN,NaN,NaN,None,None,None,False,NaN,NaN


In [5]:
if not failure_frame.empty:
    display(failure_frame.head())

raw_dataset.draw(max_num=12, caption_columns=["source_file_name", "image_format", "width", "height"])